# E1 và E2: Dummy, Logistic Regression, Decision Tree

Phần việc của người 1. Notebook làm ba việc, mỗi việc dùng đúng quy ước chung của nhóm:

1. **E1:** baseline, Logistic Regression và Decision Tree. Tinh chỉnh tham số trên tập kiểm định, **chọn theo PR-AUC** (tiêu chí chung của nhóm cho dữ liệu mất cân bằng).
2. **E2:** với từng mô hình, so sánh *không xử lý* với `class_weight="balanced"` và SMOTE.
3. Ghi kết quả vào `reports/results/<AUTHOR>.csv`, lưu xác suất vào `models/` để người khác dùng cho E7, E8, SHAP.

**Quy tắc:** chỉ dùng `train` và `val`; mọi bước học số liệu nằm trong Pipeline; không mở `test`.

In [ ]:
import sys; sys.path.insert(0, "..")   # để import được thư mục src/

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from src.data import load_xy, SEED
from src.features import build_preprocessor, check_columns
from src.evaluate import evaluate, evaluate_by_group, log_result, save_proba, MODELS_DIR

AUTHOR = "tuan"   # tên người làm; kết quả ghi vào reports/results/tuan.csv
SHOW = ["precision", "recall", "f1", "roc_auc", "pr_auc", "accuracy"]

## Đọc dữ liệu

In [ ]:
X_train, y_train = load_xy("train")
X_val, y_val = load_xy("val")
check_columns(X_train)
print(f"train {X_train.shape}, tỷ lệ hủy {y_train.mean():.1%}")
print(f"val   {X_val.shape}, tỷ lệ hủy {y_val.mean():.1%}")

## Hàm dùng chung trong notebook

`tune` thử từng tổ hợp tham số, huấn luyện trên train, chấm trên val và chọn tổ hợp có PR-AUC cao nhất. Vì val được dùng để chọn tham số nên điểm val hơi lạc quan; con số cuối cùng sẽ lấy trên test, một lần.

In [ ]:
def tune(make_model, grid):
    """Trả về (bảng kết quả sắp theo PR-AUC, tổ hợp tham số tốt nhất)."""
    rows, best = [], None
    for params in ParameterGrid(grid):
        m = evaluate(y_val, make_model(**params).fit(X_train, y_train).predict_proba(X_val)[:, 1])
        rows.append({**params, "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "f1": m["f1"]})
        if best is None or m["pr_auc"] > best[0]:
            best = (m["pr_auc"], params)
    return pd.DataFrame(rows).sort_values("pr_auc", ascending=False).reset_index(drop=True), best[1]


def run(name, model, experiment, note="", params=""):
    """Fit trên train, chấm trên val, ghi kết quả và lưu xác suất. Trả về (mô hình, chỉ số)."""
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    m = evaluate(y_val, proba)
    log_result(AUTHOR, experiment, name, "val", {**m, "params": params}, note=note)
    save_proba(name + (f"_{note}" if note else ""), proba, split="val")
    return model, m

## Baseline

Mô hình luôn đoán "không hủy". Accuracy cao nhưng recall bằng 0, nên mọi mô hình khác phải được đặt cạnh mốc này.

In [ ]:
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
m_dummy = evaluate(y_val, dummy.predict_proba(X_val)[:, 1])
log_result(AUTHOR, "E1", "dummy", "val", {**m_dummy, "params": ""})
pd.Series(m_dummy)[SHOW].round(3)

## E1: Logistic Regression

Tham số chỉnh: `C` (càng nhỏ càng phạt mạnh, mô hình càng đơn giản). Dùng bộ tiền xử lý `"linear"` (chuẩn hóa số, one-hot cho chữ).

In [ ]:
def make_lr(C=1.0, class_weight=None):
    return Pipeline([
        ("prep", build_preprocessor("linear")),
        ("clf", LogisticRegression(C=C, max_iter=2000, class_weight=class_weight, random_state=SEED)),
    ])

lr_table, lr_best = tune(make_lr, {"C": [0.001, 0.01, 0.1, 1, 10]})
print("Tham số tốt nhất:", lr_best)
lr_table.round(4)

In [ ]:
lr_model, m_lr = run("logreg", make_lr(**lr_best), "E1", params=str(lr_best))
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(lr_model, MODELS_DIR / "logreg.joblib")   # file này bị .gitignore, chia sẻ riêng cho nhóm
pd.Series(m_lr)[SHOW].round(3)

Hệ số của Logistic Regression. Các biến số đã được chuẩn hóa nên hệ số tính cho mỗi **1 độ lệch chuẩn** tăng lên; biến one-hot tính cho "có giá trị này so với không". `odds_ratio` là hệ số nhân lên **odds** (tỷ lệ hủy so với không hủy), không phải xác suất. Chỉ là "liên quan tới", không phải "gây ra".

In [ ]:
names = lr_model.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr_model.named_steps["clf"].coef_[0], index=names)
top = pd.concat([coef.nlargest(8), coef.nsmallest(8)])
top.to_frame("hệ số").assign(odds_ratio=lambda d: np.exp(d["hệ số"])).round(3)

## E1: Decision Tree

Tham số chỉnh: `max_depth` và `min_samples_leaf`. Cây không giới hạn sẽ thuộc lòng tập huấn luyện (điểm train gần tuyệt đối, điểm val thấp). Dùng bộ tiền xử lý `"tree"` (số giữ nguyên, chữ mã hóa thứ tự).

In [ ]:
def make_tree(max_depth=None, min_samples_leaf=1, class_weight=None):
    return Pipeline([
        ("prep", build_preprocessor("tree")),
        ("clf", DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                                       class_weight=class_weight, random_state=SEED)),
    ])

tree_table, tree_best = tune(make_tree, {"max_depth": [3, 5, 8, 12, None],
                                         "min_samples_leaf": [1, 20, 50, 100]})
print("Tham số tốt nhất:", tree_best)
tree_table.head(8).round(4)

In [ ]:
tree_model, m_tree = run("tree", make_tree(**tree_best), "E1", params=str(tree_best))
joblib.dump(tree_model, MODELS_DIR / "tree.joblib")
pd.Series(m_tree)[SHOW].round(3)

Các đặc trưng cây dùng nhiều nhất (độ quan trọng của cây hay thiên về cột có nhiều giá trị, nên chỉ để tham khảo; SHAP ở phần của người 3 sẽ kiểm tra lại):

In [ ]:
imp = pd.Series(tree_model.named_steps["clf"].feature_importances_,
                index=tree_model.named_steps["prep"].get_feature_names_out())
imp.nlargest(10).round(3).to_frame("độ quan trọng")

## Tổng hợp E1

In [ ]:
e1 = pd.DataFrame({"baseline": m_dummy, "logreg": m_lr, "tree": m_tree}).T[SHOW].round(3)
e1

Chỉ số tách theo khách sạn cho mô hình có PR-AUC cao hơn (yêu cầu báo cáo theo nhóm, không chỉ con số tổng):

In [ ]:
best_name, best_model = ("logreg", lr_model) if m_lr["pr_auc"] >= m_tree["pr_auc"] else ("tree", tree_model)
print("Mô hình tốt hơn theo PR-AUC:", best_name)
evaluate_by_group(y_val, best_model.predict_proba(X_val)[:, 1], X_val["hotel"])[SHOW + ["n", "positive_rate"]].round(3)

## E2: xử lý mất cân bằng lớp

Tỷ lệ hủy chỉ khoảng 24% trong tập huấn luyện. Với mỗi mô hình ở trên, giữ nguyên tham số tốt nhất của E1 rồi so sánh:

- **none:** không xử lý.
- **class_weight=balanced:** phạt nặng hơn khi đoán sai lớp hiếm. Một tham số, không thêm dữ liệu.
- **SMOTE:** tạo thêm mẫu "hủy" tổng hợp. Chỉ áp dụng **bên trong Pipeline, trên tập huấn luyện**; tập kiểm định không bị đụng tới. SMOTE nội suy giữa các điểm nên cần đầu vào dạng số liên tục, vì vậy dùng bộ tiền xử lý one-hot cho cả hai mô hình ở dòng SMOTE.

Với cây có thêm dòng **none (one-hot)** làm đối chứng, để biết chênh lệch của SMOTE đến từ SMOTE hay đến từ việc đổi cách mã hóa.

So sánh chủ yếu bằng **PR-AUC** (không phụ thuộc ngưỡng) và recall, precision, F1 ở ngưỡng 0,5. Không dùng accuracy.

In [ ]:
def smote_pipe(clf):
    return ImbPipeline([("prep", build_preprocessor("linear")),
                        ("smote", SMOTE(random_state=SEED)),
                        ("clf", clf)])

lr_clf = lambda **kw: LogisticRegression(C=lr_best["C"], max_iter=2000, random_state=SEED, **kw)
tree_clf = lambda **kw: DecisionTreeClassifier(max_depth=tree_best["max_depth"],
                                               min_samples_leaf=tree_best["min_samples_leaf"],
                                               random_state=SEED, **kw)

# dòng "none" chính là kết quả E1; ghi thêm một bản mang nhãn E2 để bảng gom E2 của cả nhóm đủ dòng
log_result(AUTHOR, "E2", "logreg", "val", {**m_lr, "params": str(lr_best)}, note="none")
log_result(AUTHOR, "E2", "tree", "val", {**m_tree, "params": str(tree_best)}, note="none")

variants = {
    ("logreg", "none"): m_lr,
    ("logreg", "balanced"): run("logreg", make_lr(**lr_best, class_weight="balanced"), "E2", "balanced", str(lr_best))[1],
    ("logreg", "SMOTE"): run("logreg", smote_pipe(lr_clf()), "E2", "SMOTE", str(lr_best))[1],
    ("tree", "none"): m_tree,
    ("tree", "none (one-hot)"): run("tree", Pipeline([("prep", build_preprocessor("linear")), ("clf", tree_clf())]),
                                    "E2", "none_onehot", str(tree_best))[1],
    ("tree", "balanced"): run("tree", make_tree(**tree_best, class_weight="balanced"), "E2", "balanced", str(tree_best))[1],
    ("tree", "SMOTE"): run("tree", smote_pipe(tree_clf()), "E2", "SMOTE", str(tree_best))[1],
}
e2 = pd.DataFrame(variants).T[SHOW].round(3)
e2.index.names = ["mô hình", "xử lý mất cân bằng"]
e2

**Đọc kết quả E2** (điền sau khi chạy):
- PR-AUC thay đổi bao nhiêu giữa các cách? Nếu gần như không đổi, xử lý mất cân bằng chủ yếu dịch chuyển điểm cắt (đổi precision lấy recall), không làm mô hình phân biệt tốt hơn.
- Recall và precision ở ngưỡng 0,5 đổi thế nào?
- **Lưu ý cho E8 (chi phí):** `class_weight` và SMOTE làm xác suất dự đoán bị đẩy lên cao hơn tần suất hủy thật. Muốn dùng xác suất để tính chi phí overbooking thì phải hiệu chỉnh lại (E7) hoặc dùng mô hình không xử lý rồi chọn ngưỡng theo chi phí.

## Kết quả đã lưu

- `reports/results/<AUTHOR>.csv`: các dòng E1 và E2 ở trên (chạy lại không bị lặp dòng).
- `models/proba_val_*.npy`: xác suất trên tập kiểm định của từng mô hình và biến thể, theo đúng thứ tự dòng của `val.csv`.
- `models/logreg.joblib`, `models/tree.joblib`: mô hình đã huấn luyện (không lên Git, gửi riêng cho người làm SHAP và app).

## Việc tiếp theo
- Điền phần "Đọc kết quả E2" và viết 3-5 câu nhận xét E1 cho báo cáo (mô hình nào tốt hơn, tại sao, sai ở nhóm nào).
- Gom kết quả E2 của cả 4 người thành một bảng chung (đọc 4 file `reports/results/*.csv`, lọc `experiment == "E2"`).
- E3 (dòng trùng) và E4 (`adr`, `special_requests`) ở notebook riêng.